In [ ]:
import warnings
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
warnings.filterwarnings('ignore')


def _parse_exs_timestamp(raw):
    if not raw:
        return None
    fixed = re.sub(r'\s+([+-]\d{2}:\d{2})$', r'\1', raw.strip())
    return datetime.fromisoformat(fixed)


def _parse_utilisation(raw, index):
    if raw is None:
        return None
    try:
        util = float(raw)
    except (ValueError, TypeError):
        s = str(raw).strip()
        if '/' in s:
            parts = s.split('/', 1)
            try:
                util = float(parts[0]) / float(parts[1])
            except (ValueError, ZeroDivisionError):
                raise ValueError(f"Execution #{index}: unparseable resource_utilisation={raw!r}")
        else:
            raise ValueError(f"Execution #{index}: unparseable resource_utilisation={raw!r}")
    if not (0.0 <= util <= 1.0):
        raise ValueError(f"Execution #{index}: resource_utilisation={util} out of [0,1]")
    return util


def parse_exs(path):
    with open(path) as f:
        data = json.load(f)
    if "executions" not in data:
        raise ValueError(f"No 'executions' key found in {path}")
    sample = data["executions"][0] if data["executions"] else {}
    is_new_format = "resource_utilisation_fired_transition" in sample
    rows = []
    for i, e in enumerate(data["executions"]):
        if is_new_format:
            util_raw = e.get("resource_utilisation_fired_transition")
        else:
            util_raw = e.get("resource_utilisation")
        util = _parse_utilisation(util_raw, i)
        fired      = int(e["fired_transition"])
        others     = list(e.get("other_enabled_transitions") or [])
        choice_set = sorted(set([fired] + others))
        activity   = e.get("activity")
        is_silent  = activity is None
        is_decision_point = (not is_silent) and (len(others) > 0)
        alt_utils = {}
        if is_new_format:
            alt_utils_raw = e.get("other_enabled_transitions_resource_utilisation") or []
            for t_id, u_raw in zip(others, alt_utils_raw):
                alt_utils[int(t_id)] = _parse_utilisation(u_raw, i)
        row = {
            "case":                 int(e["trace"]),
            "activity":             activity,
            "transition":           fired,
            "other_enabled":        others,
            "choice_set":           choice_set,
            "n_alternatives":       len(choice_set),
            "is_decision_point":    is_decision_point,
            "is_silent":            is_silent,
            "resource":             e.get("resource"),
            "resource_utilisation": util,
            "also_in_log":          bool(e.get("also_in_log", False)),
            "time_of_enablement":   _parse_exs_timestamp(e.get("time_of_enablement")),
            "time_of_execution":    _parse_exs_timestamp(e.get("time_of_execution")),
        }
        if is_new_format:
            row["alt_utils"]                = alt_utils
            row["move_index"]               = e.get("move_index")
            row["move_index_of_enablement"] = e.get("move_index_of_enablement")
        rows.append(row)
    df = pd.DataFrame(rows)
    df["case"]       = df["case"].astype("int32")
    df["transition"] = df["transition"].astype("int32")
    df["activity"]   = df["activity"].astype("category")
    df["resource"]   = df["resource"].astype("category")
    for _col in ["time_of_execution", "time_of_enablement"]:
        df[_col] = pd.to_datetime(df[_col], utc=True, errors="coerce")
    return df

In [ ]:
MINERS     = ["dfg-miner", "flw-miner", "im-miner", "imf-miner"]
INPUT_BASE = "input/3-executions-per-miner"

In [ ]:
# ── Per-miner, per-file distributions ─────────────────────────────────────────
for miner in MINERS:
    input_dir = os.path.join(INPUT_BASE, miner)
    exs_files = sorted(
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(".exs")
    )

    print(f"\n{'='*70}")
    print(f"MINER: {miner}")
    print(f"{'='*70}")

    for file_path in exs_files:
        df = parse_exs(file_path)
        dp = df[df["is_decision_point"] & df["resource_utilisation"].notna()].copy()

        if dp.empty:
            print(f"  {os.path.basename(file_path)} — no decision points, skipping")
            continue

        all_alt_vals = []
        if "alt_utils" in df.columns:
            all_alt_vals = [
                v for d in dp["alt_utils"] if isinstance(d, dict)
                for v in d.values() if v is not None
            ]

        title = f"[{miner}]  {os.path.basename(file_path)}"
        if all_alt_vals:
            title += (
                f"\nAlt utils — {len(all_alt_vals):,} values  "
                f"min={min(all_alt_vals):.4f}  max={max(all_alt_vals):.4f}  mean={np.mean(all_alt_vals):.4f}"
            )

        fig, axes = plt.subplots(1, 2, figsize=(12, 3))
        fig.suptitle(title, fontsize=8, x=0.01, ha="left", y=1.02)

        dp["resource_utilisation"].hist(bins=40, ax=axes[0], color="steelblue", edgecolor="white")
        axes[0].set_title("resource_utilisation — fired transition")
        axes[0].set_xlabel("Utilisation"); axes[0].set_ylabel("Count")

        if all_alt_vals:
            axes[1].hist(all_alt_vals, bins=40, color="tomato", edgecolor="white")
            axes[1].set_title("resource_utilisation — alternatives (all)")
            axes[1].set_xlabel("Utilisation"); axes[1].set_ylabel("Count")
        else:
            axes[1].axis("off")

        plt.tight_layout()
        plt.show()
        plt.close()

In [ ]:
# ── Combined distributions across all miners and files ────────────────────────
all_fired_vals = []
all_alt_vals   = []

for miner in MINERS:
    input_dir = os.path.join(INPUT_BASE, miner)
    exs_files = sorted(
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(".exs")
    )
    for file_path in exs_files:
        df = parse_exs(file_path)
        dp = df[df["is_decision_point"] & df["resource_utilisation"].notna()].copy()
        if dp.empty:
            continue
        all_fired_vals.extend(dp["resource_utilisation"].dropna().tolist())
        if "alt_utils" in df.columns:
            all_alt_vals.extend(
                v for d in dp["alt_utils"] if isinstance(d, dict)
                for v in d.values() if v is not None
            )

print(f"Total fired-transition util values : {len(all_fired_vals):,}")
print(f"Total alternative util values      : {len(all_alt_vals):,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(
    f"Combined — all miners, all files\n"
    f"Fired: n={len(all_fired_vals):,}  mean={np.mean(all_fired_vals):.4f}   "
    f"Alt: n={len(all_alt_vals):,}  mean={np.mean(all_alt_vals):.4f}",
    fontsize=9, x=0.01, ha="left", y=1.02,
)

axes[0].hist(all_fired_vals, bins=60, color="steelblue", edgecolor="white")
axes[0].set_title("resource_utilisation — fired transition (all datasets)")
axes[0].set_xlabel("Utilisation"); axes[0].set_ylabel("Count")

if all_alt_vals:
    axes[1].hist(all_alt_vals, bins=60, color="tomato", edgecolor="white")
    axes[1].set_title("resource_utilisation — alternatives (all datasets)")
    axes[1].set_xlabel("Utilisation"); axes[1].set_ylabel("Count")
else:
    axes[1].axis("off")

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ── Combined distributions — feasible logs only ───────────────────────────────
FEASIBLE_LOGS = {
    "BPI Challenge 2017 - Offer log.xes.gz",
    "BPI_Challenge_2013_closed_problems.xes.gz",
    "BPI_Challenge_2013_incidents.xes.gz",
    "BPI_Challenge_2013_open_problems.xes.gz",
    "bpic12-a.xes.gz",
    "bpic18 Parcel document.xes.gz",
}

def _max_norm_height(vals, bins=60):
    """Max normalized bar height for a list of utilisation values."""
    if not vals:
        return 0.0
    counts, _ = np.histogram(vals, bins=bins, range=(0, 1))
    return (counts / len(vals)).max()

def save_util_dist(fired_vals, alt_vals, label, out_path, ylim=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    title = (
        f"{label}\n"
        f"Executed: n={len(fired_vals):,}  mean={np.mean(fired_vals):.4f}   "
        f"Alternatives: n={len(alt_vals):,}  mean={np.mean(alt_vals):.4f}"
    ) if fired_vals else label
    fig.suptitle(title, fontsize=9, x=0.01, ha="left", y=1.02)

    w_fired = np.ones(len(fired_vals)) / len(fired_vals) if fired_vals else None
    axes[0].hist(fired_vals, bins=60, range=(0, 1), weights=w_fired,
                 color="steelblue", edgecolor="white")
    axes[0].set_title("resource_utilisation — executed activities")
    axes[0].set_xlabel("Utilisation"); axes[0].set_ylabel("Proportion")
    if ylim is not None:
        axes[0].set_ylim(0, ylim)

    if alt_vals:
        w_alt = np.ones(len(alt_vals)) / len(alt_vals)
        axes[1].hist(alt_vals, bins=60, range=(0, 1), weights=w_alt,
                     color="tomato", edgecolor="white")
        axes[1].set_title("resource_utilisation — alternatives")
        axes[1].set_xlabel("Utilisation"); axes[1].set_ylabel("Proportion")
        if ylim is not None:
            axes[1].set_ylim(0, ylim)
    else:
        axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()
    print(f"Saved → {out_path}")


def save_util_dist_diff(fired_vals, alt_vals, label, out_path, bins=60):
    """Bar chart of fired-normalised minus alt-normalised proportion per bin."""
    bin_edges = np.linspace(0, 1, bins + 1)
    fired_norm = np.histogram(fired_vals, bins=bin_edges)[0] / len(fired_vals)
    alt_norm   = np.histogram(alt_vals,   bins=bin_edges)[0] / len(alt_vals)
    diff    = fired_norm - alt_norm
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    width   = bin_edges[1] - bin_edges[0]
    colors  = ["steelblue" if d >= 0 else "tomato" for d in diff]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(centers, diff, width=width, color=colors, edgecolor="white", linewidth=0.3)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("Executed − Alternatives (normalised proportions) "
                 "blue → executed larger  |  red → alternatives larger")
    ax.set_xlabel("Utilisation")
    ax.set_ylabel("Δ Proportion")
    fig.suptitle(label, fontsize=9, x=0.01, ha="left", y=1.02)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()
    print(f"Saved → {out_path}")


def _log_base(fname):
    """Strip .exs and miner suffix to get the base log name."""
    stem = fname[:-4] if fname.endswith(".exs") else fname
    return re.sub(r"-(dfg|flw|im|imf|spl)$", "", stem)

feasible_fired_vals = []
feasible_alt_vals   = []

for miner in MINERS:
    input_dir = os.path.join(INPUT_BASE, miner)
    exs_files = sorted(
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(".exs") and _log_base(f) in FEASIBLE_LOGS
    )
    for file_path in exs_files:
        df = parse_exs(file_path)
        dp = df[df["is_decision_point"] & df["resource_utilisation"].notna()].copy()
        if dp.empty:
            continue
        feasible_fired_vals.extend(dp["resource_utilisation"].dropna().tolist())
        if "alt_utils" in df.columns:
            feasible_alt_vals.extend(
                v for d in dp["alt_utils"] if isinstance(d, dict)
                for v in d.values() if v is not None
            )

print(f"Total executed util values  (feasible logs): {len(feasible_fired_vals):,}")
print(f"Total alternative util values (feasible logs): {len(feasible_alt_vals):,}")

# ── Inline preview (normalised) ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(
    f"Executed: n={len(feasible_fired_vals):,}  mean={np.mean(feasible_fired_vals):.4f}   "
    f"Alternatives: n={len(feasible_alt_vals):,}  mean={np.mean(feasible_alt_vals):.4f}"
    if feasible_fired_vals else "Feasible logs only — no data found",
    fontsize=9, x=0.01, ha="left", y=1.02,
)
w_f = np.ones(len(feasible_fired_vals)) / len(feasible_fired_vals)
axes[0].hist(feasible_fired_vals, bins=60, range=(0, 1), weights=w_f,
             color="steelblue", edgecolor="white")
axes[0].set_title("resource_utilisation — executed activities")
axes[0].set_xlabel("Utilisation"); axes[0].set_ylabel("Proportion")

if feasible_alt_vals:
    w_a = np.ones(len(feasible_alt_vals)) / len(feasible_alt_vals)
    axes[1].hist(feasible_alt_vals, bins=60, range=(0, 1), weights=w_a,
                 color="tomato", edgecolor="white")
    axes[1].set_title("resource_utilisation — alternatives")
    axes[1].set_xlabel("Utilisation"); axes[1].set_ylabel("Proportion")
else:
    axes[1].axis("off")

plt.tight_layout()
plt.show()
plt.close()

# ── Save PDFs ─────────────────────────────────────────────────────────────────
_bpic17_path = os.path.join(INPUT_BASE, "dfg-miner",
                             "BPI Challenge 2017 - Offer log.xes.gz-dfg.exs")
_df17    = parse_exs(_bpic17_path)
_dp17    = _df17[_df17["is_decision_point"] & _df17["resource_utilisation"].notna()]
_fired17 = _dp17["resource_utilisation"].dropna().tolist()
_alt17   = [
    v for d in _dp17["alt_utils"] if isinstance(d, dict)
    for v in d.values() if v is not None
] if "alt_utils" in _df17.columns else []

# shared y-axis ceiling for overall + bpic17
_shared_ylim = max(
    _max_norm_height(feasible_fired_vals), _max_norm_height(feasible_alt_vals),
    _max_norm_height(_fired17),            _max_norm_height(_alt17),
) * 1.05

save_util_dist(
    feasible_fired_vals, feasible_alt_vals,
    "Distribution",
    "output/util_dist_overall.pdf",
    ylim=_shared_ylim,
)
save_util_dist(
    _fired17, _alt17,
    "BPI Challenge 2017 - Offer log   dfg-miner",
    "output/util_dist_bpic17o_dfg.pdf",
    ylim=_shared_ylim,
)
save_util_dist_diff(
    feasible_fired_vals, feasible_alt_vals,
    "Distribution — Executed minus Alternatives",
    "output/util_dist_overall_diff.pdf",
)


In [ ]:

plt.close()